# 9.6 Final Stage (Expanded)

Completes motion and disables control.

---

```python id="8k3m2x"
else:
    if not self.done:
        print("\nMotion complete. Robot should be stable at rest.")
    self.done = True
    enable_value = 0.0

self.low_cmd.motor_cmd[G1JointIndex.kNotUsedJoint].q = enable_value

self.low_cmd.crc = self.crc.Crc(self.low_cmd)
self.pub.Write(self.low_cmd)
```

---

## 🧠 Big Picture: What Is This Stage Doing?

This stage tells the robot:

```text id="v1f2k3"
"All motion is complete. Fully release SDK control and stop commanding the robot."
```

---

## ⏱️ When Does This Execute?

This is the **fallback branch**:

```text id="g4k9p2"
t ≥ 6d
```

---

### Timeline recap:

```text id="h8s2n1"
0–3d   → Stage 1 (stabilize)
3–6d   → Stage 2 (raise)
6–9d   → Stage 3 (extend)
9–15d  → Stage 4 (return)
15–18d → Stage 5 (release)
≥18d   → Final stage ✅
```

---

## 🔹 Step 1: Print Completion Message (Once)

```python id="p3d7s9"
if not self.done:
    print("\nMotion complete. Robot should be stable at rest.")
```

---

### Why check `self.done`?

Because the control loop is still running at **50 Hz**.

Without this check:

```text id="u9c2k4"
The message would print 50 times per second forever
```

---

### With this check:

```text id="w2m6q8"
Message prints exactly once
```

---

## 🔹 Step 2: Mark Controller as Done

```python id="k7n2c5"
self.done = True
```

---

### Purpose:

This signals to the **main program loop**:

```python id="z6v1y4"
if ctrl.done:
    sys.exit(0)
```

---

### 🧠 Meaning:

```text id="b5x1r2"
The controller has finished its job
```

---

## 🔹 Step 3: Fully Disable SDK Control

```python id="n8f4q1"
enable_value = 0.0
```

---

### Meaning:

```text id="t4y9k2"
SDK no longer controls the robot
```

---

### Compare with Stage 5:

| Stage       | enable_value        |
| ----------- | ------------------- |
| Stage 5     | gradually decreases |
| Final Stage | fully OFF (0.0) ✅   |

---

## 🔌 Step 4: Apply Enable Flag

```python id="j1m9s4"
self.low_cmd.motor_cmd[G1JointIndex.kNotUsedJoint].q = enable_value
```

---

### This writes to:

```text id="r2k8p6"
Special control channel (index 29)
```

---

### Effect:

```text id="y7v3n5"
Robot switches back to internal controller
```

---

## 🔐 Step 5: Compute CRC

```python id="c9x2m7"
self.low_cmd.crc = self.crc.Crc(self.low_cmd)
```

---

### Why is this required?

The robot firmware expects:

```text id="f6n1d8"
Validated command packets
```

---

### Without CRC:

* Command is rejected
* No effect on robot

---

### 🧠 Analogy:

```text id="k4w7z2"
Like a checksum in networking
```

---

## 📤 Step 6: Send Final Command

```python id="s8y3v1"
self.pub.Write(self.low_cmd)
```

---

### This publishes:

```text id="d2m5k7"
Final control message with enable = 0
```

---

### Important:

Even though control is disabled:

```text id="h3t8w9"
You must still send this command to communicate that state
```

---

## ⚠️ Subtle but Critical Behavior

After this point:

* Your control loop is still running
* But the robot is **ignoring your commands**

---

### This is intentional:

```text id="p6r2n4"
Separation of execution vs authority
```

---

## 🧠 System-Level Interpretation

This stage implements:

> **Controller termination and authority release**

---

### Before:

```text id="v8k3q1"
Your code = active controller
```

---

### After:

```text id="t9x5m2"
Robot internal system = active controller
```

---

## 🔄 Relationship to Main Loop

Your main program:

```python id="q4m7n2"
while True:
    if ctrl.done:
        sys.exit(0)
```

---

### So this stage triggers:

```text id="u2k9p5"
Program termination
```

---

## 🤖 RL Interpretation

This corresponds to:

```text id="l3f8v6"
Episode termination
```

---

### Equivalent RL concept:

```python id="y5n2r8"
done = True
```

---

### Meaning:

* Task completed
* Environment can reset
* New episode can begin

---

## ⚠️ Why Not Just Stop the Thread?

You might think:

```text id="x7p1k3"
"Why not just stop ControlLoop?"
```

---

### Because:

```text id="g9n4t6"
Robot must be explicitly told:
→ "I am releasing control"
```

---

### Otherwise:

* Robot may stay in last commanded state
* Undefined behavior may occur

---

## 🔄 Physical Interpretation

This feels like:

```text id="w6v2p9"
Letting go of a tool AFTER placing it safely
```

---

Not:

```text id="b3t8k1"
Dropping it mid-air
```

---

## 🔥 Hidden Engineering Insight

This stage demonstrates:

> **Graceful shutdown of a real-time control system**

---

Which includes:

* stopping motion
* releasing authority
* ensuring safe final state

---

## 🧠 Teaching Insight

This is a great place to emphasize:

> “Stopping a robot is just as important as moving it.”

---

Students should understand:

* Systems must terminate cleanly
* Control must be handed off safely
* Final states must be well-defined

---

## 🚀 Summary

This stage:

| Step            | Role                  |
| --------------- | --------------------- |
| Print message   | Notify completion     |
| Set `done`      | Signal program exit   |
| Disable control | Release SDK authority |
| Apply CRC       | Ensure valid message  |
| Publish command | Execute shutdown      |

---

### Result:

```text id="z1k4p7"
Robot is stable, no longer controlled by SDK, and ready for next operation
```

---

> 🔥 This completes the **full control lifecycle**: initialize → act → release → terminate.

